# DQN on a Second Game: Pong

CSCI 6353 · Topic 36 (assignment).

The previous topic built a Nature-style DQN and pointed it at Breakout, where it worked. That is *one* game. The claim that made the Nature paper famous was much stronger: **the same algorithm, network architecture, and hyperparameters, across a set of 49 games.**

This notebook does the smallest honest version of that experiment. We take the Breakout code and **change one string** so it runs on **Pong**. Nothing else about the architecture is touched. Then — and this is the assignment — **you** pick your own Atari game and reproduce the same pipeline.

The interesting part is that the string change was *not* enough on its own. The algorithm transferred; the hyperparameters did not. This notebook mirrors `train_final.py`, the actual script that trained the Pong agent in the lecture, and is honest about the one knob that decided the outcome.

## Your assignment

1. **Pick an Atari game** other than Breakout or Pong (for example `ALE/SpaceInvaders-v5`, `ALE/Enduro-v5`, `ALE/Seaquest-v5`, `ALE/BeamRider-v5`, `ALE/Qbert-v5`).
2. **Change one string** — the environment id in `make_env` below — and run *exactly* this pipeline: the same preprocessing (grayscale, 84×84, 4-frame stack, reward clipping), the same Nature architecture, and the same training loop.
3. **Report the learning curve** (episode return vs. steps) and describe what the agent learned at an early, a middle, and a late checkpoint.
4. **If it fails, find the one knob.** Do not silently retune everything. Change one hyperparameter at a time and report which one mattered, the way the lecture did for Pong (see below). An honest “it did not transfer, and here is the controlled comparison that found why” is worth more than a lucky clean run.

Two games is not 49. Be careful at the end about what an $n = 2$ (or $n = 3$) result actually shows — the point of the exercise is to *feel* how much care went into a single hyperparameter set that spans 49 games.

## 0. Colab setup

Run this once. It installs Gymnasium with the Atari (ALE) environments and PyTorch is already present on Colab. Use a **GPU runtime** (Runtime → Change runtime type → GPU) — training on CPU is impractically slow.

In [ ]:
# Colab: install the Atari environments. (Skip the pip line if you already have them.)
!pip -q install "gymnasium[atari]" ale-py

import os, sys, csv, time, random
import numpy as np
import torch, torch.nn as nn, torch.optim as optim
import gymnasium as gym
import ale_py

gym.register_envs(ale_py)
dev = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| device:", dev)

## 1. The environment and preprocessing

This is identical to the Breakout notebook. `AtariPreprocessing` does the standard pipeline: 30 no-ops at reset for stochastic starts, 4-frame skip, grayscale, and a resize to **84×84**. `FrameStackObservation(env, 4)` stacks the last four frames so the network can see motion (a single frame cannot tell you which way the ball is going).

**The entire diff from Breakout is the environment id string.** Breakout offers 4 actions, Pong offers 6, and because the code reads `env.action_space.n` instead of hard-coding it, the only structural consequence is that the output layer grows from 4 units to 6 (1,686,180 → 1,687,206 parameters, a 0.06% difference). Everything before that last layer is byte-for-byte the same network.

> **For your assignment:** change `ENV_ID` below and nothing else in this cell.

In [ ]:
ENV_ID = "ALE/Pong-v5"   # <-- the whole diff. For your game, change ONLY this string.

def make_env():
    env = gym.make(ENV_ID, frameskip=1, repeat_action_probability=0.0)
    env = gym.wrappers.AtariPreprocessing(
        env, noop_max=30, frame_skip=4, screen_size=84,
        terminal_on_life_loss=False, grayscale_obs=True, scale_obs=False)
    return gym.wrappers.FrameStackObservation(env, 4)   # 4 x 84 x 84 uint8 states

env = make_env()
N_ACT = env.action_space.n
print(f"{ENV_ID}: {N_ACT} actions, observation shape {env.observation_space.shape}")

### A note on what Pong quietly breaks

Three implementation details the Breakout notebook treated as essential do **nothing** on Pong — a useful reminder of which details were really game-specific:

- **Reward clipping is a no-op.** We clip every reward to $\pm 1$ (the `np.sign(r)` line in the loop). Pong's rewards are already exactly $+1$ (you score) and $-1$ (you concede), so the line runs and changes nothing.
- **Episodic-life is inert.** Pong has no lives, so `terminal_on_life_loss` never fires; a training episode and an evaluation episode are the same thing, one full game to 21.
- **Half the action space is redundant.** Pong's six actions collapse into three behaviours (NOOP/FIRE do nothing, RIGHT/RIGHTFIRE move one way, LEFT/LEFTFIRE the other). Nobody tells the agent this; it must discover it from experience.

Unlike Breakout, Pong also **punishes** — conceding a point costs $-1$ immediately — which makes its learning signal friendlier, and its score is a *difference* between two players, so the learning curve has a readable moment where it crosses **zero**.

## 2. The network (Nature 2015 DQN)

Three convolutional layers followed by two dense layers. Inputs are `uint8` frames scaled to $[0, 1]$ inside `forward`. This is unchanged from the Breakout notebook.

In [ ]:
class QNet(nn.Module):
    """Nature 2015 architecture: three conv layers, then two dense layers."""
    def __init__(self, n_act):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, 8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, 4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, 3, stride=1), nn.ReLU(), nn.Flatten())
        self.head = nn.Sequential(nn.Linear(64*7*7, 512), nn.ReLU(), nn.Linear(512, n_act))
    def forward(self, x):
        return self.head(self.conv(x.float() / 255.0))

_probe = QNet(N_ACT)
n_params = sum(p.numel() for p in _probe.parameters())
print(f"{ENV_ID}: {n_params:,} parameters ({N_ACT}-unit output layer)")

## 3. Hyperparameters — and the one that mattered

These mirror `train_final.py`. Every value here is **identical to the Breakout settings** except one: `TARGET_EVERY`.

The first Pong run used Breakout's target-update period (2,000 gradient steps) and **failed**. After 1.8M steps the agent scored about $-20.9$ — worse than random — and its Q-values had collapsed to a near-constant $\approx -1.03$ for every action, so the policy froze on a single action (`RIGHTFIRE`, 100% of greedy steps). Early Pong under random play concedes almost every point, so nearly every reward in the buffer is $-1$, and a constant function genuinely is the best fit to that data.

A controlled three-way comparison (1.2M steps each) found exactly **one** decisive knob:

| variant | change from Breakout settings | score at 1.2M steps |
|---|---|---|
| Breakout's settings | none | -20.9 |
| **`target`** | **target update period 2,000 → 10,000** | **-10.6** (best game +13) |
| `lr` | target fix **and** learning rate 1e-4 → 2.5e-4 | -20.9 |
| `explore` | target fix **and** $\epsilon$ floor 0.1 over 1M steps | -20.9 |

Only the variant that changed **nothing else** worked: syncing the target network every **10,000** gradient steps (Nature's value) instead of 2,000 kept the target still enough that the value function could separate the actions. Adding a larger learning rate *or* more exploration on top of that fix broke it again.

That is the honest lesson of this topic: the algorithm transferred, but a single hyperparameter chosen while looking at Breakout did not.

In [ ]:
# --- hyperparameters (Nature DQN), mirroring train_final.py ---
GAMMA, BATCH   = 0.99, 32
BUFFER         = 200_000     # uint8 replay: 84*84*4 bytes per state, x2 for s and s'
LEARN_START    = 20_000
TRAIN_EVERY    = 4
LR             = 1e-4
TARGET_EVERY   = 10_000      # <-- the decisive knob for Pong (Breakout used 2,000)
EPS_START, EPS_END, EPS_DECAY = 1.0, 0.05, 600_000
SEED           = 0

# Full lecture run was 2.5M steps (~79 min on a GPU). Start small to sanity-check.
TOTAL          = 2_500_000

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 4. The training loop

A flat, single-file DQN loop, the same one that trained the lecture's agent: an $\epsilon$-greedy step into the environment, a uint8 replay buffer, a mini-batch update every 4 steps with a Huber (smooth-L1) loss and gradient clipping, and a periodic sync of the target network. Episode returns are logged so you can plot the learning curve.

**Honesty about cost:** the full 2.5M-step run took about **79 minutes on a GPU** in the lecture. Below you can set `TOTAL` low (e.g. 50,000–100,000 steps) for a **sanity check** that everything runs end-to-end and the loss goes down — it will *not* produce a competent agent. Do a full run for real results.

In [ ]:
# ---- SANITY-CHECK SWITCH ----
# Set SHORT_RUN = True to confirm the pipeline runs in a couple of minutes.
# This will NOT learn to play; use a full run (SHORT_RUN = False) for real results.
SHORT_RUN = True
if SHORT_RUN:
    TOTAL, LEARN_START, EPS_DECAY = 50_000, 5_000, 40_000
    print(f"SHORT_RUN: {TOTAL:,} steps — sanity check only, not a trained agent.")
else:
    print(f"FULL RUN: {TOTAL:,} steps — expect ~hours (the lecture's 2.5M took ~79 min on a GPU).")

# Checkpoints let you watch the agent at three stages, as the lecture did.
CKPT = {"early": min(100_000, TOTAL), "mid": TOTAL // 2, "late": TOTAL}
OUT = "pong_out"; os.makedirs(OUT, exist_ok=True)

In [ ]:
# ---- build the network, target network, optimizer, and replay buffer ----
q, q_t = QNet(N_ACT).to(dev), QNet(N_ACT).to(dev)
q_t.load_state_dict(q.state_dict())
opt = optim.Adam(q.parameters(), lr=LR)

S  = np.zeros((BUFFER, 4, 84, 84), np.uint8)   # states s
S2 = np.zeros((BUFFER, 4, 84, 84), np.uint8)   # next states s'
A  = np.zeros(BUFFER, np.int64)
R  = np.zeros(BUFFER, np.float32)
D  = np.zeros(BUFFER, np.float32)
ptr = size = 0

def act(state, eps):
    """epsilon-greedy action."""
    if random.random() < eps:
        return random.randrange(N_ACT)
    with torch.no_grad():
        t = torch.from_numpy(np.asarray(state)).unsqueeze(0).to(dev)
        return int(q(t).argmax(1).item())

In [ ]:
# ---- the main loop ----
returns = []                       # (step, episode, return) for the learning curve
env = make_env()
s, _ = env.reset(seed=SEED)
ep_ret, ep, grad_steps, t0 = 0.0, 0, 0, time.time()

for step in range(1, TOTAL + 1):
    eps = max(EPS_END, EPS_START - (EPS_START - EPS_END) * step / EPS_DECAY)
    a = act(s, eps)
    s2, r, term, trunc, _ = env.step(a)
    done = term or trunc

    # store the transition; np.sign(r) is the reward clipping (a no-op on Pong)
    S[ptr] = np.asarray(s); S2[ptr] = np.asarray(s2)
    A[ptr] = a; R[ptr] = np.sign(r); D[ptr] = float(term)
    ptr = (ptr + 1) % BUFFER; size = min(size + 1, BUFFER)

    ep_ret += r
    s = s2
    if done:
        ep += 1
        returns.append((step, ep, ep_ret))
        ep_ret = 0.0
        s, _ = env.reset()

    # ---- learn ----
    if step > LEARN_START and step % TRAIN_EVERY == 0:
        idx = np.random.randint(0, size, BATCH)
        bs  = torch.from_numpy(S[idx]).to(dev)
        bs2 = torch.from_numpy(S2[idx]).to(dev)
        ba  = torch.from_numpy(A[idx]).to(dev)
        br  = torch.from_numpy(R[idx]).to(dev)
        bd  = torch.from_numpy(D[idx]).to(dev)
        with torch.no_grad():
            y = br + GAMMA * (1 - bd) * q_t(bs2).max(1)[0]         # TD target
        pred = q(bs).gather(1, ba.unsqueeze(1)).squeeze(1)
        loss = nn.functional.smooth_l1_loss(pred, y)              # Huber loss
        opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(q.parameters(), 10.0)
        opt.step()
        grad_steps += 1
        if grad_steps % TARGET_EVERY == 0:                        # sync target net
            q_t.load_state_dict(q.state_dict())

    # ---- checkpoints ----
    for name, at in CKPT.items():
        if step == at:
            torch.save(q.state_dict(), f"{OUT}/pong_{name}.pt")
            print(f"[{name}] checkpoint at step {step:,}", flush=True)

    # ---- progress ----
    if step % 10_000 == 0:
        recent = [r for _, _, r in returns][-20:]
        m = np.mean(recent) if recent else float("nan")
        print(f"step {step:,}  eps {eps:.3f}  last20 return {m:6.1f}  "
              f"({(time.time()-t0)/60:.1f} min)", flush=True)

torch.save(q.state_dict(), f"{OUT}/pong_final.pt")
print("DONE")

## 5. The learning curve

Plot the episode return against the training step. On a full Pong run the curve starts near $-21$ and crosses **zero** at the moment the agent begins winning more points than it concedes. In the lecture's full run the 20-game average crossed zero at **1,544,516 steps**, and the checkpoints scored:

| checkpoint | steps | score |
|---|---|---|
| early | 100,000 | -20.3 |
| mid | 1,200,000 | -8.3 |
| late | 2,500,000 | **+9.0** |

The best game was a **21–0 shutout**. (These numbers are from the lecture note; your short sanity run will look nothing like this — it is only meant to confirm the code runs.)

In [ ]:
import matplotlib.pyplot as plt

if returns:
    steps = [s for s, _, _ in returns]
    rets  = [r for _, _, r in returns]
    # rolling mean over the last 20 episodes
    k = min(20, len(rets))
    roll = [np.mean(rets[max(0, i-k+1):i+1]) for i in range(len(rets))]
    plt.figure(figsize=(8, 4))
    plt.plot(steps, rets, lw=0.5, alpha=0.4, label="episode return")
    plt.plot(steps, roll, lw=2, label=f"{k}-episode mean")
    plt.axhline(0, color="k", lw=0.8, ls="--")
    plt.xlabel("training step"); plt.ylabel("game score")
    plt.title(f"DQN on {ENV_ID}"); plt.legend(); plt.grid(alpha=0.3); plt.show()
else:
    print("No completed episodes yet — run longer.")

## 6. What two games prove (and what they do not)

One algorithm, unmodified in architecture, learned two games with different rules, action sets, reward structures, and failure modes. That is the thing DQN was celebrated for. But it needed **one hyperparameter changed** to do it, and we only found which one through a controlled comparison.

What this does **not** show is generality. Two games (or three, once you add yours) is an anecdote. The Nature paper ran 49, and the same agent that beats the human benchmark on Breakout and Pong scores **0** on Montezuma's Revenge, where the first reward needs a long, precise action sequence that random exploration essentially never stumbles into. DQN generalizes across games that **share a structure** — dense-ish rewards, short horizons between action and consequence, and a reactive skill readable from the last four frames — and fails outright where that structure is absent. Feeling how easily your own transfer can break is the real takeaway of the assignment.